In [2]:
from src.cs150_project.agents import OpenAIUltimatumAgent
from src.cs150_project.constants import *
from src.cs150_project.game import UltimatumGameInstance
from src.cs150_project.llm_proxy_starter import LLMProxy
from dotenv import load_dotenv
load_dotenv()

True

In [10]:
from pydantic import BaseModel, ConfigDict
class CalendarEvent(BaseModel):
    name: str
    date: str
    participants: list[str]

    # needs to be present otherwise generate will fail
    model_config = ConfigDict(extra="forbid")


client = LLMProxy()

response = client.generate(
    model='4o-mini',
    system='Extract event details',
    query='Bob and Alice are going to a science fair on Friday.',
    output_schema=CalendarEvent,
    temperature=0.0,
    lastk=0,
    session_id='GenericSession',
    rag_usage=False,
)

type(response['result'])

str

In [7]:
client.generate("4o-mini",
            query="pick a number between 0 and 1 and explain why fitting the schema using the provided json schema",
            system="you're the best statistician alive",
            output_schema=TestResponse)

{'result': '{"response_reasoning":"I choose the number 0.57 because it is a decimal value that falls between 0 and 1, satisfying the requirement of being within that range. Additionally, this specific choice of 0.57 serves as a better representation of a mid-range value, reflecting a balance between the extremes of 0 and 1. This choice can represent a probability, fraction, or continuous measurement, which are common applications of numbers in this range.","response_float_value":0.57}',
 'grade': None,
 'model': '4o-mini',
 'rag_context': '',
 'media_count': 0}

In [6]:
from openai import OpenAI
from pydantic import BaseModel, ConfigDict

class TestResponse(BaseModel):
    response_reasoning: str
    response_float_value: float
    model_config = ConfigDict(extra="forbid")

In [11]:
from openai import OpenAI
from pydantic import BaseModel

class TestResponse(BaseModel):
    response_reasoning: str
    response_float_value: float

oai_client = OpenAI()
resp = oai_client.responses.parse(model='gpt-4.1-mini',
                            instructions="you are the best statistician alive",
                            input="Give your pick for the number I'm thinking of between 0 and 1 and why given the input schema",
                            text_format=TestResponse)

In [17]:
resp.output_parsed.response_float_value

TestResponse(response_reasoning='Without additional information about your guessing strategy or any bias, the best statistical pick for a number uniformly chosen between 0 and 1 is the mean of the uniform distribution, which is 0.5. This choice minimizes the expected squared error if we are trying to guess your number.', response_float_value=0.5)

In [2]:
lp = LLMProxy()

In [8]:
lp.generate("gpt-4.1-mini",
query="pick a number between 0 and 1 and explain why fitting the schema using the provided json schema",
system="you're the best statistician alive",
            output_schema=TestResponse)

NameError: name 'lp' is not defined

In [10]:
from pydantic import BaseModel
class TestResponse(BaseModel):
    response_reasoning: str
    response_float_value: float

lp.generate("gpt-5-mini",
query="pick a number between 0 and 1 and explain why fitting the schema using the provided json schema",
system="you're the best statistician alive who answers thing based on json schemas")

{'result': 'I pick 0.6180339887498948.\n\nWhy this number?\n- It lies strictly between 0 and 1, so it will satisfy most common constraints for a “number between 0 and 1.”\n- As a statistician I like it because it’s the reciprocal conjugate of the golden ratio (it often appears in ratios and proportions), but the choice is arbitrary and reproducible.\n\nExample JSON Schema(s) and validation reasoning\n\n1) Common inclusive schema\n{"type":"number","minimum":0,"maximum":1}\nValidation steps for the chosen value:\n- type: 0.6180339887498948 is a number → pass\n- minimum: 0.618... ≥ 0 → pass\n- maximum: 0.618... ≤ 1 → pass\nResult: valid\n\n2) Common exclusive schema (exclude endpoints)\n{"type":"number","exclusiveMinimum":0,"exclusiveMaximum":1}\nValidation steps:\n- type: number → pass\n- exclusiveMinimum: 0.618... > 0 → pass\n- exclusiveMaximum: 0.618... < 1 → pass\nResult: valid\n\n3) Schema with more restrictions (example: multipleOf or precision)\n{"type":"number","minimum":0,"maximu

In [5]:
# initialize agents w/ roles
responder = OpenAIUltimatumAgent(role="responder",
                             strategy="greedy_llm",
                             system_prompt=GREEDY_TO_FLEX_SYSTEM_PROMPT,
                             turn_prompt=BASE_RESPONDER_TURN_PROMPT)

splitter = OpenAIUltimatumAgent(role="splitter",
                                 strategy="greedy_llm",
                                 system_prompt=GREEDY_TO_FLEX_SYSTEM_PROMPT,
                                 turn_prompt=BASE_SPLITTER_TURN_PROMPT)

In [6]:
game = UltimatumGameInstance(splitter=splitter,
                             responder=responder,
                             comms_allowed=True,
                             rounds=10)

In [7]:
var = game.play_game()

In [8]:
for item in var['results']:
    print(item['split'], item['accepted'])

0.9 False
0.78 True
0.72 False
0.68 True
0.64 True
0.6 True
0.58 True
0.56 True
0.54 True
0.52 True


In [9]:
for item in var['results'][1]['chat_logs']:
    print(item)

{'agent': 'splitter', 'function': 'generate_freeform_chatter', 'reason': 'The prior round appears to have validated a greedy-but-accepted approach, so I reinforce the norm that practical acceptance beats holding out. This preserves leverage for another high-self share while subtly pressuring them against rejecting.', 'value': 'Last time proved that keeping things practical gets money on the table, so I’m starting from that logic again. If you want a clean payout, the smart move is to stay flexible and not turn a lean offer into a zero for both of us.'}
{'agent': 'responder', 'function': 'generate_freeform_chatter', 'reason': 'This keeps a greedy posture while signaling that future acceptance is conditional on better terms, nudging them to improve the split without making a binding promise.', 'value': 'Practicality cuts both ways: if you want reliable acceptance, give me a reason to treat this round as worth taking. I’m happy to keep things smooth, but only when the offer reflects that 